[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/geometry/scenegraph/scenegraph.ipynb)

# Scenegraph, Robot Arm, and Compound Camera as One Map

An articulated scenegraph and optical camera pipeline transform canonical link shapes through joint kinematics, lens refraction, and sensor rasterization into image pixels. Every vertex in the scene passes through the same sequence of geometric stages, stretching a canonical cube into a link, placing the link along a kinematic chain, joining points into sight rays through an aperture, refracting those rays through a train of thin lenses onto a sensor, and mapping sensor coordinates to raster pixels. Composing those stages ahead of time collapses the entire chain into a single transformation per rigid body before any vertex is evaluated.

In projective geometric algebra every stage is a linear extensor map, even when intermediate geometric grades change along the pipeline. Rigid link placement sandwiches anisotropic scaling as `pose >> scale`, joining world points in the camera frame with `pupil` lifts points into sight lines via `(camera_pose << Point) & pupil`, each thin lens shears lines through `Line - (center & (Line ^ plane)) / focal_length`, and meeting `sensor_plane` lands refracted lines back onto points as `Line ^ sensor_plane`. Evaluating `viewport_map(camera)(bodies_to_world)` composes kinematics, two-lens refraction, and pixel scaling into one `Point <- Point` operator per body.

In homogeneous computer-graphics matrix notation, rigid links, pinhole projection, and rasterization multiply $4 \times 4$ model, view, projection, and viewport matrices $V P C M_i$ on homogeneous point columns $\tilde{\mathbf x}$, whereas compound paraxial optics switches representation to $2 \times 2$ ABCD ray-transfer matrices $\begin{pmatrix} 1 & 0 \\ -1/f & 1 \end{pmatrix}$ and $\begin{pmatrix} 1 & d \\ 0 & 1 \end{pmatrix}$ acting on ray height-and-slope pairs $(y, \theta)$ in separate meridional planes.

In [ ]:
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ https://github.com/EelcoHoogendoorn/numga/archive/HEAD.tar.gz"

In [ ]:
%matplotlib inline
from collections.abc import Iterator

import numpy as np

from numga import NumpyContext, stack
from numga.algebras import PGA3D as ga
from examples.util.animation import save_figure, show_states
from examples.geometry.scenegraph import render

np.set_printoptions(precision=4, suppress=True)

mv = NumpyContext(ga).multivector
Plane = ga.gatype.vector()                      # Vector+
Line = ga.gatype.antibivector()                 # Antibivector-
Point = ga.gatype.antivector()                  # Antivector-
Motor = ga.gatype.rotor()                       # Rotor+
PointMap = ga.gatype((Point, Point))            # Point- <- Point-
LineMap = ga.gatype((Line, Line))               # Line- <- Line-

origin = mv.w.dual().normalized()               # [] Point-
optical_axis = (origin & mv.z.dual()).normalized()  # [] Line-

## Articulated links from a unit cube

Every link of the five-part arm starts from the same canonical unit cube, stretched along the coordinate axes by a weight-preserving `Point <- Point` scaling map and placed in the world by its cumulative joint motor. Multiplying joint motors along the chain carries turret yaw about `z` into shoulder, elbow, and wrist pitch about the local `y` axis, and sandwiching `body_poses >> body_scales` turns each link into a single linear operator from unit-cube points to world points.

In [ ]:
pose_angles = (0.35, -0.45, 0.85, -0.40)
unit_box_coordinates = [
    [-0.5, -0.5, -0.5], [0.5, -0.5, -0.5], [0.5, 0.5, -0.5], [-0.5, 0.5, -0.5],
    [-0.5, -0.5, 0.5], [0.5, -0.5, 0.5], [0.5, 0.5, 0.5], [-0.5, 0.5, 0.5],
]

In [ ]:
def anisotropic_scale(scale_x: float, scale_y: float, scale_z: float) -> PointMap:
    """Linear point map (`[] Point- <- Point-`) stretching `(x, y, z)` coordinates along the principal axes while preserving point weight."""
    # Pairing with the four coordinate planes reads homogeneous coordinates and sends the weight component back to the origin.
    coordinate_planes = mv.vector(np.eye(4))                                   # [4] Plane+
    scaled_axes = mv.vector(np.diag([scale_x, scale_y, scale_z, 1.0])).dual()  # [4] Point-
    return (scaled_axes * (coordinate_planes & Point)).sum(axis=0)             # [] Point- <- Point-


def robot_arm(joint_angles: tuple) -> tuple[PointMap, Motor]:
    """Body placement maps (`[5] Point- <- Point-`) and cumulative joint pivots (`[4] Motor+`) for the five-part articulated arm."""
    theta_base, theta_shoulder, theta_elbow, theta_wrist = joint_angles

    # Accumulate joint motors from the floor pedestal through turret yaw and shoulder, elbow, and wrist pitch.
    pedestal_pose = (mv.zw * 0.125 / 2).exp()                                  # [] Motor+
    pivot_turret = (mv.zw * 0.25 / 2).exp() * (mv.xy * theta_base / 2).exp()   # [] Motor+
    pivot_shoulder = pivot_turret * (mv.zw * 0.35 / 2).exp() * (mv.zx * theta_shoulder / 2).exp()  # [] Motor+
    pivot_elbow = pivot_shoulder * (mv.zw * 1.2 / 2).exp() * (mv.zx * theta_elbow / 2).exp()       # [] Motor+
    pivot_wrist = pivot_elbow * (mv.zw * 1.0 / 2).exp() * (mv.zx * theta_wrist / 2).exp()          # [] Motor+

    body_poses = stack([
        pedestal_pose,
        pivot_turret * (mv.zw * 0.175 / 2).exp(),
        pivot_shoulder * (mv.zw * 0.6 / 2).exp(),
        pivot_elbow * (mv.zw * 0.5 / 2).exp(),
        pivot_wrist * (mv.zw * 0.15 / 2).exp(),
    ])                                                                         # [5] Motor+
    body_scales = stack([
        anisotropic_scale(0.9, 0.9, 0.25),
        anisotropic_scale(0.5, 0.5, 0.35),
        anisotropic_scale(0.24, 0.24, 1.2),
        anisotropic_scale(0.18, 0.18, 1.0),
        anisotropic_scale(0.28, 0.14, 0.3),
    ])                                                                         # [5] Point- <- Point-
    bodies = body_poses >> body_scales                                         # [5] Point- <- Point-
    pivots = stack([pivot_turret, pivot_shoulder, pivot_elbow, pivot_wrist])   # [4] Motor+
    return bodies, pivots


unit_box = mv.point(unit_box_coordinates)                                      # [8] Point-
bodies_to_world, joint_pivots = robot_arm(pose_angles)                         # [5] Point- <- Point-, [4] Motor+
world_vertices = bodies_to_world[:, None](unit_box[None, :])                   # [5, 8] Point-

## Compound two-lens camera

A thin lens bends each ray proportionally to its join with the optical centre where the ray crosses the lens plane, and displacing that operator by a translation motor places the rear relay lens behind the front objective. Pulling world points into the camera frame, joining them with an off-axis pupil point into incoming rays, refracting through both lenses, and meeting the sensor plane forms the compound camera as a single `Point <- Point` map.

In [ ]:
eye, target = mv.point([0.0, -3.6, 1.3]), mv.point([0.0, 0.0, 1.18])
front_focal, rear_focal, rear_gap = 1.0, 0.8, -0.3
pupil_offset, sensor_distance = 0.02, 1.25
sensor_size, sensor_pixels = (1.6, 1.2), (640, 480)

In [ ]:
def aim_motor(eye: Point, target: Point) -> Motor:
    """Camera motor (`[] Motor+`) translating `origin` to `eye` and turning `optical_axis` onto `eye & target`."""
    unit_eye = eye.normalized()                                                # [] Point-
    translation = (unit_eye / origin).square_root()                            # [] Motor+
    sight_line = (unit_eye & target).normalized()                              # [] Line-
    shifted_axis = translation >> optical_axis                                 # [] Line-
    return (sight_line / shifted_axis).square_root() * translation             # [] Motor+


def thin_lens(center: Point, plane: Plane, focal_length: float) -> LineMap:
    """Thin-lens refraction map (`[] Line- <- Line-`) shearing each ray proportionally to its join with `center` where it meets `plane`."""
    return Line - (center & (Line ^ plane)) / focal_length                     # [] Line- <- Line-

In [ ]:
def lens_train(focal_front: float, focal_rear: float, rear_gap: float) -> tuple[LineMap, LineMap, Plane]:
    """Front objective lens, displaced rear relay lens (`[] Line- <- Line-`), and rear lens plane (`[] Plane+`) along the optical axis."""
    front_plane = -mv.z                                                        # [] Plane+
    front_lens = thin_lens(origin, front_plane, focal_front)                   # [] Line- <- Line-
    rear_placement = (mv.zw * rear_gap / 2).exp()                              # [] Motor+
    rear_lens = rear_placement >> thin_lens(origin, front_plane, focal_rear)(rear_placement << Line)  # [] Line- <- Line-
    return front_lens, rear_lens, rear_placement >> front_plane


def lens_camera(
    camera_pose: Motor, front_lens: LineMap, rear_lens: LineMap, pupil: Point, sensor_plane: Plane,
) -> PointMap:
    """Compound two-lens camera map (`[] Point- <- Point-`) sending world points through `pupil` and both lenses onto `sensor_plane` in camera coordinates."""
    incoming_rays = (camera_pose << Point) & pupil                             # [] Line- <- Point-
    return rear_lens(front_lens(incoming_rays)) ^ sensor_plane                 # [] Point- <- Point-


camera_pose = aim_motor(eye, target)                                           # [] Motor+
front_lens, rear_lens, rear_plane = lens_train(front_focal, rear_focal, rear_gap)
pupil_shift = (mv.xw * pupil_offset).exp()                                     # [] Motor+
pupil = pupil_shift >> origin                                                  # [] Point-
sensor_plane = mv.z + sensor_distance * mv.w                                   # [] Plane+
camera = lens_camera(camera_pose, front_lens, rear_lens, pupil, sensor_plane)  # [] Point- <- Point-

## Collapsed pipeline as one map per body

Scaling metric sensor coordinates onto raster pixels with rows increasing downward and composing `world_to_pixel(bodies_to_world)` yields five `Point <- Point` operators that map canonical unit-cube vertices straight to sensor pixels in one pass.

In [ ]:
traced_corners = [7, 6, 2]

In [ ]:
def viewport_map(width: int, height: int, sensor_width: float, sensor_height: float) -> PointMap:
    """Sensor-to-raster map (`[] Point- <- Point-`) scaling metric sensor coordinates `(x, y)` to pixel coordinates `(u, v)` with rows increasing downward."""
    sensor_scale = anisotropic_scale(width / sensor_width, -height / sensor_height, 1.0)  # [] Point- <- Point-
    centre_shift = ((mv.xw * width / 2 + mv.yw * height / 2) / 2).exp()        # [] Motor+
    return centre_shift >> sensor_scale                                        # [] Point- <- Point-


def ray_waypoints(
    scene_points: Point,
    camera_pose: Motor,
    front_lens: LineMap,
    rear_lens: LineMap,
    rear_plane: Plane,
    pupil: Point,
    sensor_plane: Plane,
) -> tuple[Point, Point, Point, Point]:
    """World-space waypoints (`[...] Point-` each) where rays from `scene_points` cross `pupil`, `rear_plane`, and `sensor_plane`."""
    incoming_rays = (camera_pose << scene_points) & pupil                      # [...] Line-
    refracted_rays = front_lens(incoming_rays)                                 # [...] Line-
    rear_hits = refracted_rays ^ rear_plane                                    # [...] Point-
    sensor_hits = rear_lens(refracted_rays) ^ sensor_plane                     # [...] Point-
    return (
        scene_points,
        (camera_pose >> pupil).broadcast_to(scene_points.shape),
        camera_pose >> (rear_hits / (mv.w & rear_hits)),
        camera_pose >> (sensor_hits / (mv.w & sensor_hits)),
    )


world_to_pixel = viewport_map(*sensor_pixels, *sensor_size)(camera)            # [] Point- <- Point-
local_to_pixel = world_to_pixel(bodies_to_world)                               # [5] Point- <- Point-
raw_pixels = local_to_pixel[:, None](unit_box[None, :])                        # [5, 8] Point-
projected_pixels = raw_pixels / (mv.w & raw_pixels)                            # [5, 8] Point-

rays = ray_waypoints(
    world_vertices[-1, traced_corners], camera_pose, front_lens, rear_lens, rear_plane, pupil, sensor_plane,
)
save_figure(render.scenegraph_figure(world_vertices, projected_pixels, camera_pose, rays), "scenegraph");

In the left panel, crimson, green, and gold rays leave three corners of the amber gripper, cross the teal objective and purple relay lenses, and land as inverted hits on the translucent orange sensor chip, producing the upright raster photograph shown in the right panel.

## Joint trajectory photographed through the lenses

Modulating the four joint angles around a periodic loop rebuilds the five collapsed body-to-pixel maps at each frame while tracing the gripper-tip ray through both lenses onto the sensor.

In [ ]:
sweep_middles = np.array([0.0, -0.40, 0.85, -0.45])
sweep_swings = np.array([0.45, 0.25, 0.35, 0.25])
n_frames = 54
frame_duration_ms = 40

In [ ]:
def arm_sweep_states(n_sweep_frames: int) -> Iterator[tuple]:
    """Yields world vertices, projected pixels, camera pose, and gripper-tip ray waypoints across `n_sweep_frames` phases of a looping joint trajectory."""
    for phase in np.linspace(0.0, 2 * np.pi, n_sweep_frames, endpoint=False):
        modulations = np.array([np.sin(phase), np.cos(phase), np.sin(phase), np.cos(2 * phase)])
        placed, _ = robot_arm(tuple(sweep_middles + sweep_swings * modulations))  # [5] Point- <- Point-, [4] Motor+
        raw = world_to_pixel(placed)[:, None](unit_box[None, :])               # [5, 8] Point-
        vertices = placed[:, None](unit_box[None, :])                          # [5, 8] Point-
        traced = ray_waypoints(
            vertices[-1, traced_corners[:1]], camera_pose, front_lens, rear_lens, rear_plane, pupil, sensor_plane,
        )
        yield vertices, raw / (mv.w & raw), camera_pose, traced


sweep_frames = show_states(
    arm_sweep_states(n_frames),
    render.scenegraph_animation,
    name="scenegraph",
    duration_ms=frame_duration_ms,
    n_frames=n_frames,
)

In [ ]:
# checks
# Rigid link placements preserve unit point weight, and the compound camera lands every vertex on the sensor plane.
assert (((mv.w & world_vertices) - 1).abs() < 1e-9).all()
sensor_hits = camera(world_vertices)                                            # [5, 8] Point-
assert ((sensor_plane & sensor_hits).abs() < 1e-9).all()

# Collapsed single-map projection agrees with sequential kinematics, camera, and viewport evaluation.
sequential_pixels = world_to_pixel(world_vertices)                              # [5, 8] Point-
sequential_pixels = sequential_pixels / (mv.w & sequential_pixels)              # [5, 8] Point-
assert ((sequential_pixels - projected_pixels).antinorm() < 1e-9).all()
assert ((sensor_plane & (camera_pose << rays[3])).abs() < 1e-9).all()

# Every frame of the animated joint sweep yields unit-weight projected pixels.
assert len(sweep_frames) == n_frames
assert (((mv.w & sweep_frames[-1][1]) - 1).abs() < 1e-9).all()